In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from scipy.optimize import minimize

from rich import print
import seaborn as sns
sns.set()

import ewatercycle
import ewatercycle.models
import ewatercycle.forcing
from scipy.stats import qmc
from ipywidgets import IntProgress

In [2]:
calibration_start_date = "1975-01-01"
calibration_end_date = "2010-12-31"

In [3]:
forcing_path_ERA5 = Path.home() / "BEP-beau/BEP/code" / "CatchmentArea" / "ERA5_3" / "own_shapefile_3"
load_location = forcing_path_ERA5 / "work" / "diagnostic" / "script"  
ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=load_location)

data = pd.read_csv("mohembo_daily_water_discharge_data.csv", index_col='date', parse_dates=True, dayfirst=True)
data_daily = data.resample('D').interpolate()
data_daily.columns = ['Discharge (m^3/s)']
print(ERA5_forcing)

LumpedMakkinkForcing(
    start_time='1970-01-01T00:00:00Z',
    end_time='2020-12-31T00:00:00Z',
    directory=PosixPath('/home/beau/BEP-beau/BEP/code/CatchmentArea/ERA5_3/own_shapefile_3/work/diagnostic/script')
,
    shape=PosixPath('/home/beau/BEP-beau/BEP/code/CatchmentArea/ERA5_3/own_shapefile_3/work/diagnostic/script/Catch
mentArea_4326.shp'),
    filenames={
        'pr': 'OBS6_ERA5_reanaly_1_day_pr_1970-2020.nc',
        'tas': 'OBS6_ERA5_reanaly_1_day_tas_1970-2020.nc',
        'rsds': 'OBS6_ERA5_reanaly_1_day_rsds_1970-2020.nc',
        'evspsblpot': 'Derived_Makkink_evspsblpot.nc'
    }
)

In [4]:
Area_km2 = 173696.852

def mmday_to_m3s(mmday_data, area):
    return (mmday_data * area) / 86.4

In [5]:
def KGE(modelled, observed, start, end):
    start = pd.to_datetime(start).tz_localize(None)
    end = pd.to_datetime(end).tz_localize(None)
    
    modelled.index = pd.to_datetime(modelled.index)
    observed.index = pd.to_datetime(observed.index)

    df = pd.concat([modelled.reindex(observed.index, method='ffill'), observed], axis=1, keys=['Modelled', 'Observed'])
    df = df.dropna()
    df = df[(df.index > start) & (df.index < end)]

    r = np.corrcoef(df['Observed'], df['Modelled'])[0, 1]
    beta = np.mean(df['Modelled']) / np.mean(df['Observed'])
    CV_modelled = np.std(df['Modelled']) / np.mean(df['Modelled'])
    CV_observed = np.std(df['Observed']) / np.mean(df['Observed'])                                 
    gamma = CV_modelled / CV_observed
    
    kge = 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)
    return kge

In [6]:
N = 1
s_0 = np.array([0,  100,  0,  5,  0])

param_names = ["Imax", "Ce", "Sumax", "Beta", "Pmax", "Tlag", "Kf", "Ks", "FM"]
param_mins = np.array([0, 0.2, 40, 0.5, 0.001, 1, 0.01, 0.0001, 0.01])
param_maxs = np.array([8, 1, 800, 4, 0.3, 10, 0.1, 0.01, 1])

sampler = qmc.LatinHypercube(d=len(param_names))
sample = sampler.random(n=N)
parameters = qmc.scale(sample, param_mins, param_maxs)

In [7]:
ensemble = []

for counter in range(N): 
    ensemble.append(ewatercycle.models.HBVLocal(forcing=ERA5_forcing))
    config_file, _ = ensemble[counter].setup(parameters = parameters[counter],  initial_storage=s_0)
    ensemble[counter].initialize(config_file)

In [8]:
f = IntProgress(min=0, max=N)
display(f)

KGE_values = []
for ensembleMember in ensemble:
    Q_m_KGE = []
    time_KGE = []
    while ensembleMember.time < ensembleMember.end_time:
        ensembleMember.update()
        discharge_this_timestep = ensembleMember.get_value("Q")
        Q_m_KGE.append(discharge_this_timestep[0])
        time_KGE.append(ensembleMember.time_as_datetime)

    Q_m_KGE = mmday_to_m3s(np.array(Q_m_KGE), Area_km2)
    discharge_dataframe = pd.DataFrame({'model output': Q_m_KGE}, index=pd.to_datetime(time_KGE))

    kgevalue = KGE(discharge_dataframe['model output'], data_daily['Discharge (m^3/s)'], calibration_start_date, calibration_end_date)
    KGE_values.append(kgevalue)
    
    del Q_m_KGE, time_KGE, discharge_dataframe, kgevalue
    f.value += 1
    
for ensembleMember in ensemble:
    ensembleMember.finalize()

IntProgress(value=0, max=1)

In [9]:
best_index = np.argmax(KGE_values)
best_parameters = parameters[best_index]
best_KGE = KGE_values[best_index]
print(list(zip(param_names, np.round(best_parameters, decimals=3))))
print(f"Best KGE: {best_KGE:.3f}")
best_parameters

[
    ('Imax', 2.426),
    ('Ce', 0.468),
    ('Sumax', 338.962),
    ('Beta', 1.627),
    ('Pmax', 0.062),
    ('Tlag', 1.494),
    ('Kf', 0.085),
    ('Ks', 0.004),
    ('FM', 0.801)
]

Best KGE: -2.900

array([2.42585719e+00, 4.68187544e-01, 3.38961623e+02, 1.62661292e+00,
       6.15476999e-02, 1.49361061e+00, 8.45055999e-02, 4.44204674e-03,
       8.00953835e-01])